In [1]:
import pandas as pd
import xgboost as xgb
import numpy as np
from sklearn.metrics import precision_recall_curve,log_loss,accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,matthews_corrcoef,roc_auc_score
import optuna
import gc
from scipy.stats import norm
from collections import Counter
from scipy.stats import norm
from functools import partial
from sklearn.model_selection import StratifiedKFold

/home/s1856122/anaconda3/envs/KinSPEC/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# set bedroc
def calculate_bedroc(y_true, y_score, bedroc_alpha):
    # y_true: true binary labels
    y_true  = np.asarray(y_true)
    # y_score: predicted scores
    y_score = np.asarray(y_score)
    N = len(y_true)
    n_act = np.sum(y_true == 1)
    if n_act == 0:
        return 0.0                    # Return 0 when there are no active compounds

    # 1. Sort predictions and extract the ranks of active compounds
    order  = np.argsort(y_score)[::-1]
    ranks  = np.arange(1, N+1)[y_true[order] == 1]      # 1-based
    x      = (ranks - 0.5) / N                          # Centered rank

    # 2. Calculate RIE
    C      = (1.0 - np.exp(-bedroc_alpha)) / bedroc_alpha
    rie    = np.mean(np.exp(-bedroc_alpha * x)) / C

    # 3. Calculate the theoretical limits
    x_max  = (np.arange(1, n_act+1) - 0.5) / N          # Best case
    x_min  = (np.arange(N-n_act+1, N+1) - 0.5) / N      # Worst case
    rie_max = np.mean(np.exp(-bedroc_alpha * x_max)) / C
    rie_min = np.mean(np.exp(-bedroc_alpha * x_min)) / C

    # 4. Normalize the score
    return (rie - rie_min) / (rie_max - rie_min)

In [3]:
# Calculate the enrichment factor
def calculate_ef(y_true, y_score, percentages=[0.5, 1.0, 2.0, 3.0,4.0, 5.0]):

    # y_true: true binary labels
    y_true = np.asarray(y_true)
    # y_score: predicted scores
    y_score = np.asarray(y_score)
    
    # Check the inputs
    if len(y_true) != len(y_score):
        raise ValueError("y_true and y_score must have the same length")
    
    N = len(y_true)  # Total number of compounds
    n_active = np.sum(y_true == 1)  # Total number of active compounds
    
    if n_active == 0:
        print("Warning: no active compounds were found; returning EF = 0")
        return {f"EF_{p}%": 0.0 for p in percentages}
    
    # Sort compounds by predicted score in descending order
    sorted_indices = np.argsort(y_score)[::-1]
    sorted_labels = y_true[sorted_indices]
    
    results = {}
    
    for percentage in percentages:
        # Calculate the number of compounds corresponding to the top x%
        n_selected = max(1, int(N * percentage / 100))  # Select at least one compound
        
        # Number of active compounds in the top x%
        n_active_in_selected = np.sum(sorted_labels[:n_selected])
        
        # Calculate the enrichment factor
        # EF = active-compound fraction in the top x% / active-compound fraction in the full dataset
        ef_value = (n_active_in_selected / n_selected) / (n_active / N)
        
        results[f"EF_{percentage}%"] = ef_value
        
        # Print detailed information
        print(f"EF_{percentage}%:")
        print(f"  - Selected compounds: {n_selected}/{N} ({percentage}%)")
        print(f"  - Active compounds among selected compounds: {n_active_in_selected}")
        print(f"  - Active-compound fraction: {n_active_in_selected/n_selected:.4f}")
        print(f"  - Overall active-compound fraction: {n_active/N:.4f}")
        print(f"  - EF value: {ef_value:.4f}")
        print()
    
    return results

In [4]:
# Define a function to test the trained model
def run_the_model_full(Xy,y,kinase_name):

    # Load the model
    final_model = xgb.Booster()
    final_model.load_model("../model/KinSPEC_full_training_pool_MCA_200D.json")
    y_pred_prob = final_model.predict(Xy)  # Predicted probabilities

    # Automatically select the optimal threshold
    precisions, recalls, thresholds = precision_recall_curve(y, y_pred_prob)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    optimal_idx = np.argmax(f1_scores)
    y_pred = (y_pred_prob > thresholds[optimal_idx])

    # Calculate common evaluation metrics
    auc = roc_auc_score(y, y_pred_prob)
    auc_var = np.var(y_pred_prob) / len(y_pred_prob)
    z = norm.ppf(1 - (1 - 0.95) / 2)  # Calculate the z value
    lower_bound = auc - z * np.sqrt(auc_var)
    upper_bound = auc + z * np.sqrt(auc_var)

    acc = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    mcc = matthews_corrcoef(y, y_pred)
    logloss = log_loss(y, y_pred_prob)

    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0.0
    npv = tn / (tn + fn) if (tn + fn) != 0 else 0.0

     # Calculate confidence intervals by bootstrapping
    n_bootstraps = 1000
    bootstrapped_aucs = []

    # Convert y and y_pred_prob to NumPy arrays
    y = np.array(y)
    y_pred_prob = np.array(y_pred_prob)

    rng = np.random.RandomState(42)
    for _ in range(n_bootstraps):
        indices = rng.choice(len(y), size=len(y), replace=True)
        auc_bootstrap = roc_auc_score(y[indices], y_pred_prob[indices])
        bootstrapped_aucs.append(auc_bootstrap)
    ci_lower = np.percentile(bootstrapped_aucs, 2.5)
    ci_upper = np.percentile(bootstrapped_aucs, 97.5)
    print(f"AUC: {auc:.3f} (95% CI: [{ci_lower:.3f}, {ci_upper:.3f}])")

    # Calculate BEDROC
    bedroc_score = calculate_bedroc(y, y_pred_prob, bedroc_alpha=80.5)
    ef_score = calculate_ef(y, y_pred_prob)
    # Print evaluation results
    print(f"Best threshold: {thresholds[optimal_idx]:.4f}")
    print(f"AUC: {auc:.4f} (95% CI: {lower_bound:.3f}, {upper_bound:.3f})")
    print(f"bedroc: {bedroc_score:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision(PPV): {precision:.4f}")
    print(f"Recall(Sensitivity): {recall:.4f}")
    print(f"F1 score: {f1:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"NPV: {npv:.4f}")
    print(f"MCC: {mcc:.4f}")
    print(f"Log Loss: {logloss:.4f}")
    print(confusion_matrix(y, y_pred))
    print(f"True Negative = {tn}")
    print(f"True Positive = {tp}")
    print(f"False Negative = {fn}")
    print(f"False Positive = {fp}")
    print("Class distribution:", Counter(y))
    print("Positive-class fraction:", np.mean(y))

    if kinase_name is not None:
        df_output = pd.DataFrame({
            "Best threshold": f"{thresholds[optimal_idx]:.4f}",
            "AUC": f"{auc:.4f}",
            "AUC 95% CI lower_bound": f"{lower_bound:.3f}",
            "AUC 95% CI upper_bound": f"{upper_bound:.3f}",
            "bedroc": f"{bedroc_score:.4f}",
            "Accuracy": f"{acc:.4f}",
            "Precision(PPV)": f"{precision:.4f}",
            "Recall(Sensitivity)": f"{recall:.4f}",
            "F1 score": f"{f1:.4f}",
            "Specificity": f"{specificity:.4f}",
            "NPV": f"{npv:.4f}",
            "MCC": f"{mcc:.4f}",
            "Log Loss": f"{logloss:.4f}",
            "True Negative": f"{tn}",
            "True Positive": f"{tp}",
            "False Negative": f"{fn}",
            "False Positive": f"{fp}"
        },index=[kinase_name])
        return bedroc_score,df_output,ef_score
    else:
        return bedroc_score

In [5]:
def klifs_map(df,KLIFS_file,KLIFS_latent): # df is the dataset to map; KLIFS_file contains all KLIFS entries; KLIFS_latent contains the corresponding reduced KLIFS features
    # Read KLIFS sequences from the input dataset
    read_klifs = df['KLIFS'].astype(str).str.strip()
    # Read KLIFS sequences from the KLIFS reference dataset
    KLIFS = KLIFS_file['KLIFS'].astype(str).str.strip()
    # Map each KLIFS sequence to its reduced latent representation
    keys = KLIFS
    values = KLIFS_latent
    KLIFS_dict = dict(zip(keys,values))
    # Perform the mapping
    kinase_result = [KLIFS_dict.get(x.lstrip(),None) for x in read_klifs]
    return kinase_result

In [9]:
# vsds test dataset
vsds_mol_fp = pd.read_csv("../dataset/vsds-vd_truedecoy/LSAE_vsds_latent_top95_features_2.csv",skiprows=1,header=None)

vsds = pd.read_csv("../dataset/vsds-vd_truedecoy/VSDS_truedecoy_integrated_data.csv")

vsds_klifs = vsds['KLIFS']

vsds_bio = vsds['molecule_label']

# mapping KLIFS to LSAE embedded KLIFS feature
KLIFS_sae_top = pd.read_csv("../LSAE_embedding/Kinase_test_MCA/MCA_klifs_482_200D_features.csv",skiprows=1, header=None).values
KLIFS_file = pd.read_csv("../LSAE_embedding/kinase/human_482_typical_kinase_1955bit.csv")

vsds_kinase_fp = pd.DataFrame(klifs_map(vsds,KLIFS_file,KLIFS_sae_top))
vsds_input_feature = pd.concat([vsds_mol_fp,vsds_kinase_fp],axis=1)

vsds_input_feature.columns = range(vsds_input_feature.shape[1])

print(vsds_mol_fp.shape)
print(vsds_kinase_fp.shape)

vsds_Xy = xgb.DMatrix(vsds_input_feature,label=vsds_bio)

(49705, 466)
(49706, 200)


In [10]:
print('evaluation metrics on VSDS test set:')
vsds_test = run_the_model_full(vsds_Xy,vsds_bio,None)

evaluation metrics on VSDS test set:
AUC: 0.661 (95% CI: [0.646, 0.676])
EF_0.5%:
  - Selected compounds: 248/49706 (0.5%)
  - Active compounds among selected compounds: 49
  - Active-compound fraction: 0.1976
  - Overall active-compound fraction: 0.0241
  - EF value: 8.1978

EF_1.0%:
  - Selected compounds: 497/49706 (1.0%)
  - Active compounds among selected compounds: 75
  - Active-compound fraction: 0.1509
  - Overall active-compound fraction: 0.0241
  - EF value: 6.2612

EF_2.0%:
  - Selected compounds: 994/49706 (2.0%)
  - Active compounds among selected compounds: 107
  - Active-compound fraction: 0.1076
  - Overall active-compound fraction: 0.0241
  - EF value: 4.4663

EF_3.0%:
  - Selected compounds: 1491/49706 (3.0%)
  - Active compounds among selected compounds: 126
  - Active-compound fraction: 0.0845
  - Overall active-compound fraction: 0.0241
  - EF value: 3.5063

EF_4.0%:
  - Selected compounds: 1988/49706 (4.0%)
  - Active compounds among selected compounds: 153
  - Ac